# AegisFlow M3 — Supervised Model Training on Kaggle CPU

This notebook trains the AegisFlow Milestone 3 supervised fraud detection models using the official IEEE-CIS Fraud Detection dataset.

### Pipeline Architecture
1. **Git Baseline**: Clones repository at approved commit `2533ccc7325c32255a707e123f49418e833aa6d3`.
2. **Dependencies**: Installs pinned dependencies from `ml/requirements-train.txt` (including DVC `3.56.0`).
3. **Dataset Discovery & Zero-Copy Symlink**: Dynamically locates `train_transaction.csv` and `train_identity.csv` in `/kaggle/input` and links `data/raw/ieee`.
4. **MLflow Configuration**: Configures local SQLite tracking (`mlflow.db`) and verifies runtime artifact locations.
5. **Execution**: Runs `dvc repro` to execute both `prepare_ieee` and `train_fraud` without altering stage semantics.
6. **Verification**: Validates all stage outputs, metrics, and models.
7. **Packaging**: Creates `aegisflow_m3_kaggle_bundle.tar.gz` with complete DVC outputs, lockfile, manifest, and MLflow model artifacts.

## Stage 1 — Clone Repository at Approved Commit Baseline

In [ ]:
import subprocess, sys, os, pathlib

REPO_URL = "https://github.com/Hariprasath1x/AegisFlow.git"
TARGET_COMMIT = "2533ccc7325c32255a707e123f49418e833aa6d3"
REPO_DIR = pathlib.Path("/kaggle/working/aegisflow").resolve()

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", "-b", "feature/m3-fraud-models", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already cloned at {REPO_DIR}.")

os.chdir(REPO_DIR)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Current HEAD: {current_head}")
if current_head != TARGET_COMMIT:
    print(f"Checking out approved commit {TARGET_COMMIT}...")
    subprocess.run(["git", "checkout", TARGET_COMMIT], check=True)
    current_head = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()

assert current_head == TARGET_COMMIT, f"Commit mismatch! Expected {TARGET_COMMIT}, got {current_head}"
print(f"Git baseline verified at approved commit: {current_head}")

## Stage 2 — Install Pinned CPU Dependencies & Verify DVC Version

In [ ]:
import os, pathlib, shutil, subprocess

PY312_BIN = "/usr/bin/python3.12"
VENV_DIR = pathlib.Path("/kaggle/working/aegisflow-py312")
VENV_PYTHON = VENV_DIR / "bin" / "python"
VENV_DVC = VENV_DIR / "bin" / "dvc"

assert pathlib.Path(PY312_BIN).exists(), f"Expected Python 3.12 at {PY312_BIN}"
subprocess.run([PY312_BIN, "--version"], check=True)

if not VENV_PYTHON.exists():
    print(f"Creating Python 3.12 virtual environment at {VENV_DIR}...")
    res = subprocess.run([PY312_BIN, "-m", "venv", str(VENV_DIR)], capture_output=True, text=True)
    if res.returncode != 0:
        print("Installing python3.12-venv via apt-get...")
        if VENV_DIR.exists():
            shutil.rmtree(VENV_DIR)
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "python3.12-venv"], check=True)
        subprocess.run([PY312_BIN, "-m", "venv", str(VENV_DIR)], check=True)

# Prepend venv bin to PATH for all subsequent subprocesses
os.environ["PATH"] = f"{VENV_DIR / 'bin'}:{os.environ.get('PATH', '')}"

print(f"Installing pinned dependencies from ml/requirements-train.txt into {VENV_PYTHON}...")
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-q", "-r", "ml/requirements-train.txt"], check=True)

verify_code = """
import sys, dvc, xgboost, lightgbm, sklearn, shap, mlflow, pyarrow, pandas, numpy
assert sys.version.startswith("3.12"), f"Unexpected Python: {sys.version}"
assert dvc.__version__.startswith("3.56"), f"Unexpected DVC version: {dvc.__version__}"
print("Python:", sys.version)
print("  dvc:", dvc.__version__)
print("  xgboost:", xgboost.__version__)
print("  lightgbm:", lightgbm.__version__)
print("  scikit-learn:", sklearn.__version__)
print("  shap:", shap.__version__)
print("  mlflow:", mlflow.__version__)
print("  pyarrow:", pyarrow.__version__)
print("  pandas:", pandas.__version__)
print("  numpy:", numpy.__version__)
"""
subprocess.run([str(VENV_PYTHON), "-c", verify_code], check=True)

## Stage 3 — Dynamic Dataset Discovery & Symlink Preflight

In [ ]:
import pathlib, os, sys, subprocess

print("Scanning /kaggle/input for IEEE-CIS Fraud Detection dataset...")
input_root = pathlib.Path("/kaggle/input")
tx_matches = list(input_root.rglob("train_transaction.csv"))

if not tx_matches:
    tx_zips = list(input_root.rglob("train_transaction.csv.zip"))
    if tx_zips:
        raise FileNotFoundError(
            f"Found compressed dataset {tx_zips[0]}. Please attach the uncompressed competition dataset."
        )
    raise FileNotFoundError("train_transaction.csv not found in /kaggle/input! Please attach ieee-fraud-detection.")

raw_dir = tx_matches[0].parent
print(f"Discovered train_transaction.csv at: {raw_dir}")

id_file = raw_dir / "train_identity.csv"
if not id_file.exists():
    raise FileNotFoundError(f"train_identity.csv missing from discovered directory: {raw_dir}")
print(f"Discovered train_identity.csv at: {id_file}")

assert tx_matches[0].stat().st_size > 100_000_000, "train_transaction.csv appears truncated"
assert id_file.stat().st_size > 10_000_000, "train_identity.csv appears truncated"

raw_target = pathlib.Path("data/raw/ieee").resolve()
raw_target.parent.mkdir(parents=True, exist_ok=True)
if raw_target.is_symlink() or raw_target.exists():
    raw_target.unlink()

raw_target.symlink_to(raw_dir.resolve())
print(f"Created symlink: {raw_target} -> {raw_dir.resolve()}")

dvc_status = subprocess.run([str(VENV_DVC), "status"], capture_output=True, text=True)
print("DVC status output:")
print(dvc_status.stdout)
assert "prepare_ieee" in dvc_status.stdout or "data/raw/ieee" in dvc_status.stdout, (
    "DVC did not recognize data/raw/ieee dependency!"
)
print("DVC dependency recognized successfully.")

## Stage 4 — Configure MLflow Tracking & Runtime Artifact Verification

In [ ]:
import os, pathlib, subprocess

db_path = pathlib.Path("/kaggle/working/aegisflow/mlflow.db").resolve()
mlruns_dir = pathlib.Path("/kaggle/working/aegisflow/mlruns").resolve()
mlruns_dir.mkdir(parents=True, exist_ok=True)

tracking_uri = f"sqlite:///{db_path}"
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri

mlflow_init = (
    "import os, mlflow\n"
    "from mlflow.tracking import MlflowClient\n"
    "uri = os.environ['MLFLOW_TRACKING_URI']\n"
    "mlflow.set_tracking_uri(uri)\n"
    "client = MlflowClient(tracking_uri=uri)\n"
    "exp = client.get_experiment('0')\n"
    "print('MLflow Tracking URI:', uri)\n"
    "print('Default Experiment ID:', exp.experiment_id)\n"
    "print('Experiment Artifact Location:', exp.artifact_location)\n"
    "print('MLflow backend and storage initialized.')\n"
)
subprocess.run([str(VENV_PYTHON), "-c", mlflow_init], check=True)

## Stage 5 — Execute Authoritative Pipeline via `dvc repro`

In [ ]:
import os, subprocess

print("Executing authoritative M3 pipeline via dvc repro (Python 3.12 + MPLBACKEND=Agg)...")
print("Executing: prepare_ieee -> train_fraud")
env = {
    **os.environ,
    "PATH": f"{VENV_DIR / 'bin'}:{os.environ.get('PATH', '')}",
    "PYTHONPATH": "ml/src:.",
    "MPLBACKEND": "Agg",
}
subprocess.run([str(VENV_DVC), "repro"], env=env, check=True)
print("dvc repro finished successfully.")

## Stage 6 — Verify All Stage Outputs & Model Artifacts

In [ ]:
import os, subprocess

verify_outputs = """
import json, pathlib, mlflow
from mlflow.tracking import MlflowClient

lock_file = pathlib.Path("dvc.lock")
assert lock_file.exists(), "dvc.lock was not created!"
lock_text = lock_file.read_text()
assert "prepare_ieee" in lock_text and "train_fraud" in lock_text, "Incomplete dvc.lock"
print("dvc.lock verified.")

parquet_path = pathlib.Path("data/processed/ieee_transactions.parquet")
meta_path = pathlib.Path("data/processed/ieee_meta.json")
assert parquet_path.exists() and parquet_path.stat().st_size > 10_000_000, "Transactions Parquet missing or empty"
assert meta_path.exists(), "ieee_meta.json missing"
ieee_meta = json.loads(meta_path.read_text())
print(f"Processed Data: {ieee_meta['rows']} rows, fraud rate {ieee_meta['fraud_rate']:.3%}")

fraud_json = pathlib.Path("reports/m3_fraud.json")
fraud_html = pathlib.Path("reports/m3_fraud.html")
assert fraud_json.exists() and fraud_html.exists(), "Evaluation reports missing"
report = json.loads(fraud_json.read_text())
print("Promotion gate evaluations:")
for g in report.get("gates", []):
    print(f"  {g.get('detail')} -> {'PASS' if g.get('pass') else 'FAIL'}")
    assert g.get("pass"), f"Promotion gate failure: {g}"

metrics_file = pathlib.Path("artifacts/fraud/metrics.json")
assert metrics_file.exists(), "artifacts/fraud/metrics.json missing"
metrics = json.loads(metrics_file.read_text())
champ = metrics["champion"]
print(f"Champion: {champ}")
print(f"Recall@1%FPR: {metrics['results'][champ]['recall_at_1pct_fpr']:.4f}")
print(f"Lift over rules: {metrics['results'][champ]['lift_over_rules']:.2%}")
print(f"ECE: {metrics['results'][champ]['ece']:.4f}")

client = MlflowClient()
runs = client.search_runs("0")
assert len(runs) > 0, "No MLflow runs found"
k_run = runs[0]
print(f"Kaggle Run ID: {k_run.info.run_id}")
print(f"Run Artifact URI: {k_run.info.artifact_uri}")

loaded_m = mlflow.sklearn.load_model(f"{k_run.info.artifact_uri}/model")
loaded_c = mlflow.sklearn.load_model(f"{k_run.info.artifact_uri}/calibrator")
print(f"Successfully loaded estimator: {type(loaded_m)}")
print(f"Successfully loaded calibrator: {type(loaded_c)}")
"""
subprocess.run([str(VENV_PYTHON), "-c", verify_outputs], env={**os.environ, "MPLBACKEND": "Agg"}, check=True)

## Stage 7 — Package Complete Export Bundle & Manifest

In [ ]:
import os, subprocess

package_script = """
import json, datetime, os, pathlib, shutil, tarfile, dvc, sys
import xgboost, lightgbm, sklearn, shap, mlflow, pandas, numpy
from mlflow.tracking import MlflowClient

target_commit = os.environ["AEGIS_TARGET_COMMIT"]
metrics = json.loads(pathlib.Path("artifacts/fraud/metrics.json").read_text())
champ = metrics["champion"]
client = MlflowClient()
runs = client.search_runs("0")
assert len(runs) > 0, "No MLflow runs found"
k_run_id = runs[0].info.run_id

export_dir = pathlib.Path("exported_mlflow")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

run_artifacts_dir = pathlib.Path(mlflow.get_tracking_uri().replace("sqlite:///", "")).parent / "mlruns" / "0" / k_run_id / "artifacts"
for item in run_artifacts_dir.iterdir():
    dest = export_dir / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)
print(f"Staged MLflow artifacts into {export_dir}")

manifest = {
    "git_commit": target_commit,
    "git_branch": "feature/m3-fraud-models",
    "kaggle_run_id": k_run_id,
    "dvc_version": dvc.__version__,
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "platform": "kaggle_cpu",
    "champion_algorithm": champ,
    "python_version": sys.version,
    "packages": {
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
        "scikit_learn": sklearn.__version__,
        "shap": shap.__version__,
        "mlflow": mlflow.__version__,
        "pandas": pandas.__version__,
        "numpy": numpy.__version__,
    },
    "metrics": {
        "recall_at_1pct_fpr": metrics["results"][champ]["recall_at_1pct_fpr"],
        "lift_over_rules": metrics["results"][champ]["lift_over_rules"],
        "pr_auc": metrics["results"][champ]["pr_auc"],
        "ece": metrics["results"][champ]["ece"],
    }
}
pathlib.Path("execution_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Created execution_manifest.json")

bundle_path = pathlib.Path("/kaggle/working/aegisflow_m3_kaggle_bundle.tar.gz")
if bundle_path.exists():
    bundle_path.unlink()

files_to_bundle = [
    "dvc.lock",
    "execution_manifest.json",
    "data/processed/ieee_transactions.parquet",
    "data/processed/ieee_meta.json",
    "reports/m3_fraud.json",
    "reports/m3_fraud.html",
    "artifacts/fraud",
    "exported_mlflow",
    "mlruns",
]

print(f"Archiving bundle to {bundle_path}...")
with tarfile.open(bundle_path, "w:gz") as tar:
    for rel_path in files_to_bundle:
        p = pathlib.Path(rel_path)
        if p.exists():
            tar.add(p, arcname=rel_path)
            print(f"  Archived: {rel_path}")
        else:
            raise FileNotFoundError(f"Required bundle file {rel_path} is missing!")

print(f"Export bundle complete: {bundle_path} ({bundle_path.stat().st_size / (1024*1024):.2f} MB)")
print("Ready for download from Kaggle Output.")
"""
subprocess.run(
    [str(VENV_PYTHON), "-c", package_script],
    env={**os.environ, "MPLBACKEND": "Agg", "AEGIS_TARGET_COMMIT": TARGET_COMMIT},
    check=True,
)